In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.1 Base vs instruction models

> 🎯 **Goal:** See why a freshly pretrained (base) model only continues text, and
> understand that the whole of this unit is about teaching it to answer.

**Why this matters.** Everything you built so far produced a *base* model: it is
very good at predicting the next token, which makes it good at continuing text in
the style of whatever it read. But continuing is not answering. A user types a
question and expects a reply, not more of the question. Unit u7 is the bridge from
a clever autocomplete to a useful assistant, and this lesson sets up the gap that
the rest of the unit closes.

**The intuition.** Picture a very well read person who has only ever been asked to
*finish your sentences*. Hand them 'The capital of France is' and they will happily
say 'Paris'. Hand them 'Question: what is your name? Answer:' and they might write
another exam question, because that is what often follows in the books they read.
They are not being unhelpful, they were simply never *taught the job of answering*.
Post-training (this unit) is the on-the-job training.

Second, a crucial correction beginners always get wrong: **context is not
memory**. Each call to the model is stateless. The model sees only the text in the
prompt right now. It has no recollection of an earlier chat, no hidden notebook,
nothing carried over between calls. When a chatbot 'remembers' your name, the app
is quietly re-sending the earlier turns inside the prompt every single time.

**The mechanics.** A **base model** is the network straight out of pretraining: its
only objective was next-token prediction over a big corpus. **Instruction
following** is an extra behavior layered on afterward through *post-training*: the
family of techniques (SFT, LoRA, QLoRA, preference optimization) you will meet in
lessons l7.2 through l7.5. **Fine-tuning** is the umbrella term for continuing to
train a pretrained model on new data so it picks up new behavior. Below, we train a
tiny base model and feed it a question to watch the continuation-not-answer gap
with our own eyes.

In [ ]:
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
out = generate(model, tok, "Question: what is your name? Answer:",
               max_new_tokens=40, seed=0)
print(out)

**What just happened.** The model took your prompt and kept writing in the style
of its training data. It did not produce a clean, helpful 'answer', because nothing
ever taught it that a line starting with 'Answer:' should be followed by a direct
reply. The output is plausible *continuation*, not a *response*. That gap is the
entire reason post-training exists.

⚠️ **Common pitfalls**
- Expecting a base model to behave like ChatGPT. Out of the box it will not, the
  helpfulness you are used to was added by SFT and preference tuning.
- Confusing context with memory. If you want the model to 'remember' something, you
  must put it in the prompt of *this* call. There is no carryover between calls.
- Reading a weird continuation as a 'bug'. A base model continuing oddly is working
  exactly as trained, the job just is not the one you wanted yet.

🏋️ **Try it yourself.** Swap the prompt for a plain continuation like
'Once upon a time' and rerun. Notice how much more natural the base model feels
when you ask it to do the one thing it was actually trained for. Then try a
two-line 'chat' (paste a fake previous turn) and confirm the model only responds to
what is in the prompt, with no memory of anything outside it.

In [ ]:
assert len(out) > len("Question: what is your name? Answer:")